# 08 - Publication Figures & Visual Storyboard Pipeline

This notebook dynamically generates the **Publication Figures** and **Thesis Visual Storyboard** across all evaluated dimensions, noise regimes, and solvers.

### Global Data Integrity & Pipeline Rules
1. **Dynamic Database Discovery**: Solvers, models, problem IDs, dimensions, and noise levels are discovered dynamically from SQLite (`data/db.sqlite3`) and IOH evaluation files (`results/evaluations/`).
2. **User Filter Controls**: Allows selective filtering by model, problem, prompt strategy, dimension, and noise level (or defaults to plotting all completed runs).
3. **Adaptive Subplot Architectures**: All multi-panel figures adapt grid geometry dynamically to the number of active problem instances.
4. **Publication Standards**: High-DPI exports (300 DPI), consistent typography (Inter/Helvetica), clean margins, and semantically mapped color palettes.

---
### Generated Figure Suites
- **Comparative Suite** (`results/publication/{dim}D/`)
  - `figure_e_difficulty_and_noise.png`: Landscape hardness success rates & Fragility Index heatmap.
  - `figure_1_benchmark_validation.png`: (RQ1) Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
  - `figure_3_robustness.png`: (RQ3) Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Model-Specific Suite** (`results/evaluations/visual_profiles/{model}/{dim}D/`)
  - `figure_4_scaffolding.png`: (RQ2/3 Ablation) Prompt Scaffolding Ablation on multi-strategy models.
  - `figure_success_rate_by_hardness.png`: Clean vs. Noisy success rates by landscape hardness.
  - `std_{noise}/convergence_trajectories.png`: Dynamic multi-panel convergence trajectories with IQR shaded bounds.
  - `std_{noise}/target_precision_ecdf.png`: Empirical target precision hit rates (ECDF).

In [19]:
%load_ext autoreload
%autoreload 2

import sqlite3
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import sys
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from shared.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR
from benchmarking import (
    StatisticalEvaluationService,
    IOHTraceReader,
    BBOB_CLASSES,
    BBOB_NAMES,
    get_bbob_class,
    get_bbob_name,
    resolve_canonical_model_slug,
    resolve_folder_solver_name,
)
parse_dat_file = IOHTraceReader.parse_dat_file

service = StatisticalEvaluationService()
EVALUATIONS_DIR   = RESULTS_DIR / 'evaluations' / 'traces'
PUBLICATION_DIR   = RESULTS_DIR / 'publication'
EVAL_PROFILES_DIR = RESULTS_DIR / 'evaluations' / 'visual_profiles'
DB_PATH         = DATA_DIR / 'db.sqlite3'
PUBLICATION_DIR.mkdir(parents=True, exist_ok=True)
EVAL_PROFILES_DIR.mkdir(parents=True, exist_ok=True)

def comparative_dir(dim: int) -> Path:
    p = PUBLICATION_DIR / f'{dim}D'
    p.mkdir(parents=True, exist_ok=True)
    return p

comparative_fig_dir = comparative_dir

def model_fig_dir(folder_name: str, dim: int) -> Path:
    slug = resolve_canonical_model_slug(folder_name)
    p = EVAL_PROFILES_DIR / slug / f'{dim}D'
    p.mkdir(parents=True, exist_ok=True)
    return p

model_dir = model_fig_dir

def model_std_dir(model_slug: str, dim: int, noise_std: float) -> Path:
    slug = resolve_canonical_model_slug(model_slug)
    p = EVAL_PROFILES_DIR / slug / f'{dim}D' / f'std_{noise_std}'
    p.mkdir(parents=True, exist_ok=True)
    return p

# ── User Execution Controls & Selective Filters ──────────────────────────────
FILTER_MODELS     = None   # e.g., ['14b'], ['7b'], or None for all
FILTER_PROBLEMS   = None   # e.g., [1, 8], [8, 11, 15], or None for all
FILTER_STRATEGIES = None   # e.g., ['baseline', 'guided'], or None for all
FILTER_DIMS       = None   # e.g., [2, 3], [5], or None for all
FILTER_NOISE_STDS = None   # e.g., [0.0, 0.05], or None for all

# Set global plotting aesthetic
plt.rcParams['font.sans-serif'] = ['Inter', 'Helvetica', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.edgecolor']   = '#CCCCCC'
plt.rcParams['axes.linewidth']   = 0.8

print('✅ Figure environment initialized.')


✅ Figure environment initialized.


In [20]:
# ── 1. Dynamic BBOB Metadata (All 24 Benchmark Functions) ───────────────────
BBOB_METADATA = {
    1:  ('Sphere', 'Separable'),
    2:  ('Ellipsoidal', 'Separable'),
    3:  ('Rastrigin', 'Separable'),
    4:  ('Buche-Rastrigin', 'Separable'),
    5:  ('Linear Slope', 'Separable'),
    6:  ('Attractive Sector', 'Low Conditioning'),
    7:  ('Step Ellipsoidal', 'Low Conditioning'),
    8:  ('Rosenbrock', 'Low Conditioning'),
    9:  ('Rosenbrock Rotated', 'Low Conditioning'),
    10: ('Ellipsoidal High-Cond', 'High Conditioning'),
    11: ('Discus', 'High Conditioning'),
    12: ('Bent Cigar', 'High Conditioning'),
    13: ('Sharp Ridge', 'High Conditioning'),
    14: ('Different Powers', 'High Conditioning'),
    15: ('Rastrigin Multi-Modal', 'Multi-Modal (Global)'),
    16: ('Weierstrass', 'Multi-Modal (Global)'),
    17: ('Schaffers F7', 'Multi-Modal (Global)'),
    18: ('Schaffers F7 Ill-Cond', 'Multi-Modal (Global)'),
    19: ('Griewank-Rosenbrock', 'Multi-Modal (Global)'),
    20: ('Schwefel', 'Multi-Modal (Weak)'),
    21: ('Gallagher 101 Peaks', 'Multi-Modal (Weak)'),
    22: ('Gallagher 21 Peaks', 'Multi-Modal (Weak)'),
    23: ('Katsuura', 'Multi-Modal (Weak)'),
    24: ('Lunacek Bi-Rastrigin', 'Multi-Modal (Weak)')
}

def get_bbob_name(p_id: int) -> str:
    name, _ = BBOB_METADATA.get(p_id, (f'Function {p_id}', 'General'))
    return f'{name} (f{p_id})'

def get_bbob_class(p_id: int) -> str:
    _, cls = BBOB_METADATA.get(p_id, (f'Function {p_id}', 'General'))
    return cls

BBOB_NAMES = {p: get_bbob_name(p) for p in range(1, 25)}
BBOB_CLASSES = {p: get_bbob_class(p) for p in range(1, 25)}

# ── 2. Dynamic Solver Palette Generator ───────────────────────────────────────
def build_dynamic_solver_palette(solvers: list[str]) -> dict[str, str]:
    """Algorithmically assign publication-grade colors to dynamically discovered solvers.
    Uses strategy-based semantic hues for LLM variants and neutral earth tones for classical baselines.
    """
    STRATEGY_COLORS = {
        'baseline':      '#1f77b4',  # Muted Blue
        'guided':        '#ff7f0e',  # Safety Orange
        'thinking':      '#2ca02c',  # Green
        'vectorization': '#d62728',  # Crimson Red
    }
    CLASSICAL_COLORS = {
        'CMA-ES': '#8c564b',  # Chestnut Brown
        'DE':     '#e377c2',  # Raspberry Pink
        'PSO':    '#7f7f7f',  # Slate Gray
    }
    QUALITATIVE_CYCLE = pc.qualitative.Plotly + pc.qualitative.Dark24 + pc.qualitative.Set1
    palette = {}
    fallback_idx = 0
    
    for s in solvers:
        if s in CLASSICAL_COLORS:
            palette[s] = CLASSICAL_COLORS[s]
        elif ' / ' in s:
            model_tag, strat = s.split(' / ', 1)
            strat_lower = strat.lower()
            if '14B' in model_tag and strat_lower in STRATEGY_COLORS:
                palette[s] = STRATEGY_COLORS[strat_lower]
            elif '7B' in model_tag and strat_lower == 'baseline':
                palette[s] = '#9467bd'  # Muted Royal Purple
            elif strat_lower in STRATEGY_COLORS and '14B' not in model_tag:
                palette[s] = QUALITATIVE_CYCLE[fallback_idx % len(QUALITATIVE_CYCLE)]
                fallback_idx += 1
            else:
                palette[s] = QUALITATIVE_CYCLE[fallback_idx % len(QUALITATIVE_CYCLE)]
                fallback_idx += 1
        else:
            palette[s] = QUALITATIVE_CYCLE[fallback_idx % len(QUALITATIVE_CYCLE)]
            fallback_idx += 1
    return palette

# ── 3. Data Parsers for Evaluations & SQLite Database ─────────────────────────
def parse_dat_file(dat_path: Path):
    runs = []
    current_evals, current_raw = [], []
    with open(dat_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith(('function', 'evaluations', '"evaluations"', '#', 'instance')):
                if current_evals:
                    runs.append((np.array(current_evals), np.array(current_raw)))
                    current_evals, current_raw = [], []
                continue
            parts = line.split()
            if len(parts) >= 2:
                try:
                    current_evals.append(float(parts[0]))
                    current_raw.append(float(parts[1]))
                except ValueError:
                    continue
    if current_evals:
        runs.append((np.array(current_evals), np.array(current_raw)))
    return runs

def resolve_solver_name(parent_name: str, prov_data: dict | None = None) -> str:
    p = parent_name.lower()
    if 'cmaes' in p or 'cma_es' in p: return 'CMA-ES'
    if 'pso' in p: return 'PSO'
    if p == 'de' or p.startswith(('de_', 'de-')) or '_de_' in p: return 'DE'
    
    if prov_data:
        model_str = prov_data.get('model_tag') or prov_data.get('llm_name', '')
        strat_str = prov_data.get('prompt_strategy', '')
        m = re.search(r'(\d+b)', str(model_str).lower())
        if m:
            model_label = f'LLaMEA-{m.group(1).upper()}'
        else:
            clean_m = str(model_str).removesuffix('.gguf').replace('llamea', '').strip('_- ').replace('_', ' ').title()
            model_label = f'LLaMEA-{clean_m}' if clean_m else 'LLaMEA'
        return f'{model_label} / {strat_str.lower()}' if strat_str else model_label
    
    parts = parent_name.split('_')
    if len(parts) >= 2:
        strat = parts[-1].lower()
        model_slug = '_'.join(parts[:-1])
        m = re.search(r'(\d+b)', model_slug.lower())
        if m:
            model_label = f'LLaMEA-{m.group(1).upper()}'
        else:
            clean_m = model_slug.replace('llamea', '').strip('_').replace('_', ' ').title()
            model_label = f'LLaMEA-{clean_m}' if clean_m else 'LLaMEA'
        return f'{model_label} / {strat}'
    return parent_name

def load_benchmark_data(eval_dir: Path, filter_models=None, filter_problems=None, filter_strategies=None, filter_dims=None, filter_noise_stds=None):
    data_store = {}
    if not eval_dir.exists(): return data_store
    for json_path in eval_dir.glob('**/*.json'):
        if json_path.name == 'provenance.json': continue
        try:
            with open(json_path, 'r', encoding='utf-8') as jf: meta = json.load(jf)
        except Exception: continue
        path_str = str(json_path.relative_to(eval_dir))
        dim_m = re.search(r'(\d+)D', path_str); dim = int(dim_m.group(1)) if dim_m else None
        noise_m = re.search(r'std_([\d\.]+)', path_str); noise_std = float(noise_m.group(1)) if noise_m else 0.0
        p_id = meta.get('function_id')
        if p_id is None: p_m = re.search(r'f(\d+)', path_str); p_id = int(p_m.group(1)) if p_m else None
        parent_dir = json_path.parent
        parent_name = parent_dir.name
        if 'dummy' in parent_name.lower(): continue
        
        prov_file = parent_dir / 'provenance.json'
        prov_data = None
        if prov_file.exists():
            try:
                with open(prov_file, 'r', encoding='utf-8') as pf: prov_data = json.load(pf)
            except Exception: pass
            
        solver_name = resolve_solver_name(parent_name, prov_data)
        
        for sc in meta.get('scenarios', []):
            sc_dim = dim if dim is not None else sc.get('dimension')
            if p_id is None or sc_dim is None: continue
            
            # Apply filters
            if filter_dims is not None and sc_dim not in filter_dims: continue
            if filter_noise_stds is not None and not any(np.isclose(noise_std, fn) for fn in filter_noise_stds): continue
            if filter_problems is not None and p_id not in filter_problems: continue
            if filter_models is not None:
                if solver_name not in ['CMA-ES', 'DE', 'PSO']:
                    if not any(fm.lower() in solver_name.lower() for fm in filter_models): continue
            if filter_strategies is not None:
                if ' / ' in solver_name:
                    strat = solver_name.split(' / ')[1].lower()
                    if strat not in [fs.lower() for fs in filter_strategies]: continue
                    
            key = (sc_dim, noise_std, p_id)
            if key not in data_store: data_store[key] = {}
            if solver_name not in data_store[key]: data_store[key][solver_name] = []
            dat_p = sc.get('path')
            if dat_p and (parent_dir / dat_p).exists():
                data_store[key][solver_name].extend(parse_dat_file(parent_dir / dat_p))
    return data_store

def load_sqlite_synthesis_data(db_path: Path):
    if not db_path.exists(): return pd.DataFrame(), pd.DataFrame()
    with sqlite3.connect(db_path) as conn:
        df_exp = pd.read_sql_query('SELECT * FROM experiments', conn)
        df_iter = pd.read_sql_query(
            'SELECT i.id AS iteration_id, i.experiment_id, i.algorithm_name, i.raw_fitness, i.final_error, '
            'i.timed_out, i.converged, i.runtime_seconds, e.problem_id, e.dim, e.mode, e.llm_name, '
            'e.prompt_strategy, e.noise_std FROM iterations i JOIN experiments e ON i.experiment_id = e.id',
            conn
        )
    def map_exp_solver(row):
        llm = str(row.get('llm_name', '')).lower()
        strat = str(row.get('prompt_strategy', '')).lower()
        m = re.search(r'(\d+b)', llm)
        if m: return f'LLaMEA-{m.group(1).upper()} / {strat}'
        clean_llm = llm.removesuffix('.gguf').replace('-', ' ').replace('_', ' ').title()
        return f'LLaMEA-{clean_llm} / {strat}'
    if not df_exp.empty:
        df_exp['solver_name'] = df_exp.apply(map_exp_solver, axis=1)
    if not df_iter.empty:
        df_iter['solver_name'] = df_iter.apply(map_exp_solver, axis=1)
    return df_exp, df_iter

print('✅ Universal BBOB metadata, palette engine, and parsers loaded successfully.')


✅ Universal BBOB metadata, palette engine, and parsers loaded successfully.


In [21]:
# ── 3. Dynamically Discover Solvers, Conditions, and Build Palette Registry ──
df_exp, df_iter = load_sqlite_synthesis_data(DB_PATH)
all_benchmark_data = load_benchmark_data(
    EVALUATIONS_DIR,
    filter_models=FILTER_MODELS,
    filter_problems=FILTER_PROBLEMS,
    filter_strategies=FILTER_STRATEGIES,
    filter_dims=FILTER_DIMS,
    filter_noise_stds=FILTER_NOISE_STDS
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark data matched the current filters in {EVALUATIONS_DIR}!')

all_dims = sorted(list(set(k[0] for k in all_benchmark_data.keys())))
all_noise_stds = sorted(list(set(k[1] for k in all_benchmark_data.keys())))
clean_std = 0.0 if 0.0 in all_noise_stds else all_noise_stds[0]
noisy_stds = [s for s in all_noise_stds if not np.isclose(s, clean_std)]
noisy_std = noisy_stds[0] if noisy_stds else (all_noise_stds[-1] if len(all_noise_stds) > 1 else clean_std)

PROBLEM_IDS = sorted(list(set(k[2] for k in all_benchmark_data.keys())))

raw_solvers = set(s for cond in all_benchmark_data.values() for s in cond.keys())
ALL_SOLVERS_ORDER = sorted(
    list(raw_solvers),
    key=lambda s: (1 if s in ['CMA-ES', 'DE', 'PSO'] else 0, s)
)

SOLVER_COLORS = build_dynamic_solver_palette(ALL_SOLVERS_ORDER)
SOLVERS_CLASSICAL = [s for s in ALL_SOLVERS_ORDER if s in ['CMA-ES', 'DE', 'PSO']]
SOLVERS_LLM = [s for s in ALL_SOLVERS_ORDER if s not in SOLVERS_CLASSICAL]

MODELS_TO_SOLVERS = defaultdict(list)
for s in SOLVERS_LLM:
    model_part = s.split(' / ')[0] if ' / ' in s else s
    MODELS_TO_SOLVERS[model_part].append(s)

DISCOVERED_MODELS = list(MODELS_TO_SOLVERS.keys())

print(f'📦 Loaded {len(all_benchmark_data)} benchmark problem conditions:')
print(f'   • Evaluated Dimensions: {all_dims}')
print(f'   • Noise Regimes: Clean σ={clean_std}, Noisy σ={noisy_std} (all: {all_noise_stds})')
print(f'   • Active BBOB Problem IDs ({len(PROBLEM_IDS)}): {PROBLEM_IDS}')
print(f'   • Evaluated Solvers ({len(ALL_SOLVERS_ORDER)}): {ALL_SOLVERS_ORDER}')
print(f'   • Model Families ({len(MODELS_TO_SOLVERS)}): {list(MODELS_TO_SOLVERS.keys())}')
print(f'   • Classical Baselines: {SOLVERS_CLASSICAL}')


📦 Loaded 30 benchmark problem conditions:
   • Evaluated Dimensions: [2, 3, 5]
   • Noise Regimes: Clean σ=0.0, Noisy σ=0.05 (all: [0.0, 0.05])
   • Active BBOB Problem IDs (5): [1, 8, 11, 15, 21]
   • Evaluated Solvers (11): ['LLaMEA-14B / baseline', 'LLaMEA-14B / guided', 'LLaMEA-14B / thinking', 'LLaMEA-14B / vectorization', 'LLaMEA-14B / vectorization-1', 'LLaMEA-7B / baseline', 'LLaMEA-7B / guided', 'LLaMEA-7B / thinking', 'LLaMEA-7B / vectorization', 'CMA-ES', 'DE']
   • Model Families (2): ['LLaMEA-14B', 'LLaMEA-7B']
   • Classical Baselines: ['CMA-ES', 'DE']


# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
clean_std = 0.0
noisy_std = 0.05

for dim in all_dims:


In [22]:
# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
for dim in all_dims:
    fig_e = make_subplots(
        rows=2, cols=2,
        specs=[[{}, {}], [{'colspan': 2}, None]],
        subplot_titles=(
            f'<b>(A1) Mean Success Rate — Clean (σ={clean_std}, {dim}D)</b>',
            f'<b>(A2) Mean Success Rate — Noisy (σ={noisy_std}, {dim}D)</b>',
            f'<b>(B) Landscape Fragility Index Matrix (Clean → Noisy σ={noisy_std} Degradation, {dim}D)</b>'
        ),
        vertical_spacing=0.18,
        horizontal_spacing=0.08,
        row_heights=[0.45, 0.55]
    )

    # Subplots A1 & A2 (Clean vs. Noisy Hardness Success Rates across Solvers)
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        prob_records = []
        for (d, n_std, p_id), solvers in all_benchmark_data.items():
            if d != dim or not np.isclose(n_std, noise_level): continue
            h_class = BBOB_CLASSES.get(p_id, 'Unknown')
            for solver_name, runs in solvers.items():
                if not runs: continue
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                if not finals: continue
                n_conv = sum(1 for f in finals if f <= 1e-8)
                sr = n_conv / len(finals) * 100.0
                prob_records.append({'hardness_class': h_class, 'solver_name': solver_name, 'success_rate': sr})
        
        df_p = pd.DataFrame(prob_records)
        df_c = df_p.groupby(['hardness_class', 'solver_name'])['success_rate'].mean().reset_index() if not df_p.empty else pd.DataFrame()
        
        # Dynamic hardness classes present in active data
        standard_classes = ['Separable', 'Low Conditioning', 'High Conditioning', 'Multi-Modal (Global)', 'Multi-Modal (Weak)']
        active_classes = [c for c in standard_classes if c in set(BBOB_CLASSES.get(p, '') for p in PROBLEM_IDS)]
        if not active_classes: active_classes = standard_classes
        
        for s in ALL_SOLVERS_ORDER:
            df_s = df_c[df_c['solver_name'] == s] if not df_c.empty else pd.DataFrame()
            if not df_s.empty:
                df_s_map = dict(zip(df_s['hardness_class'], df_s['success_rate']))
                y_vals = [df_s_map.get(c, 0.0) for c in active_classes]
            else:
                y_vals = [0.0 for _ in active_classes]
            
            fig_e.add_trace(
                go.Bar(
                    x=active_classes, y=y_vals, name=s,
                    marker_color=SOLVER_COLORS.get(s, '#7f7f7f'),
                    showlegend=(c_idx == 1 and dim == all_dims[0])
                ),
                row=1, col=c_idx
            )
        
        fig_e.update_yaxes(title='<b>Success Rate (%)</b>' if c_idx == 1 else None, range=[0, 105], row=1, col=c_idx)
        fig_e.update_xaxes(title='<b>Landscape Hardness</b>', tickangle=-15, row=1, col=c_idx)
        fig_e.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False, row=1, col=c_idx)
        fig_e.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False, row=1, col=c_idx)
    
    # Subplot B: Fragility Heatmap across all active problem IDs
    fragility_rows = []
    for s in ALL_SOLVERS_ORDER:
        s_row = []
        for p_id in PROBLEM_IDS:
            runs_clean = all_benchmark_data.get((dim, clean_std, p_id), {}).get(s, [])
            runs_noisy = all_benchmark_data.get((dim, noisy_std, p_id), {}).get(s, [])
            err_c = np.median([r[1][-1] for r in runs_clean if len(r[1]) > 0]) if runs_clean else 1e-12
            err_n = np.median([r[1][-1] for r in runs_noisy if len(r[1]) > 0]) if runs_noisy else 1e-12
            log_c = np.log10(max(err_c, 1e-12))
            log_n = np.log10(max(err_n, 1e-12))
            s_row.append(round(float(log_n - log_c), 2))
        fragility_rows.append(s_row)
    
    p_labels = [f"{BBOB_NAMES.get(p, f'f{p}')}<br>({BBOB_CLASSES.get(p, '')})" for p in PROBLEM_IDS]
    fig_e.add_trace(
        go.Heatmap(
            z=fragility_rows, x=p_labels, y=ALL_SOLVERS_ORDER,
            colorscale='Blues', text=fragility_rows, texttemplate='%{text}',
            colorbar=dict(title='Δ log₁₀(Err)', len=0.45, y=0.22, yanchor='middle', x=1.02),
            showscale=True
        ),
        row=2, col=1
    )
    fig_e.update_xaxes(title='<b>BBOB Problem Suite (Hardness Ordered)</b>', row=2, col=1)
    fig_e.update_yaxes(title='<b>Evaluated Solver</b>', row=2, col=1)
    
    fig_e.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Problem Difficulty & Noise Sensitivity Dashboard (Clean vs. Noisy) — {dim}D</b>',
            x=0.02, y=0.985,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#333333'),
        margin=dict(l=65, r=45, t=130, b=55),
        width=1150, height=920,
        legend=dict(
            orientation='h', yanchor='bottom', y=1.04, xanchor='center', x=0.5,
            bgcolor='rgba(255,255,255,0.92)', bordercolor='rgba(0,0,0,0.12)', borderwidth=1,
            font=dict(size=10.0)
        )
    )
    out_path = comparative_dir(dim) / 'figure_e_difficulty_and_noise.png'
    fig_e.write_image(str(out_path), scale=3)

print(f'✅ Figure E generated for all dimensions in {PUBLICATION_DIR}/.')


2026-08-24 00:23:59 INFO Chromium init'ed with kwargs {}
2026-08-24 00:23:59 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 00:23:59 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpuoiz6x4o.
2026-08-24 00:23:59 INFO Opening browser.
2026-08-24 00:23:59 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp62q13za1.
2026-08-24 00:23:59 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp62q13za1
2026-08-24 00:24:02 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpuoiz6x4o/index.html
2026-08-24 00:24:03 INFO Getting tab from queue (has 1)
2026-08-24 00:24:03 INFO Got 5734
2026-08-24 00:24:03 INFO Reloading tab 5734 before return.
2026-08-24 00:24:03 INFO Putting tab 5734 back (queue size: 0).
2026-08-24 00:24:03 INFO Waiting for all cleanups to finish.
2026-08-24 00:24:03 INFO Exiting Kaleido.
2026-08-24 00:24:03 INFO T

✅ Figure E generated for all dimensions in /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/figures/comparative/.


### 📊 Model-Specific Hardness Success Rates (Clean vs. Noisy)
Separates the mean success rate analysis per LLM model (, ) across clean and noisy landscapes.

In [23]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'<b>(A) Clean Landscape (σ={clean_std}, {dim}D)</b>',
            f'<b>(B) Noisy Landscape (σ={noisy_std}, {dim}D)</b>'
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        prob_records = []
        for (d, n_std, p_id), solvers in all_benchmark_data.items():
            if d != dim or not np.isclose(n_std, noise_level): continue
            h_class = BBOB_CLASSES.get(p_id, 'Unknown')
            for solver_name, runs in solvers.items():
                if solver_name not in solvers_list or not runs: continue
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                if not finals: continue
                n_conv = sum(1 for f in finals if f <= 1e-8)
                sr = n_conv / len(finals) * 100.0
                prob_records.append({'hardness_class': h_class, 'solver_name': solver_name, 'success_rate': sr})
        
        df_p = pd.DataFrame(prob_records)
        df_c = df_p.groupby(['hardness_class', 'solver_name'])['success_rate'].mean().reset_index() if not df_p.empty else pd.DataFrame()
        
        standard_classes = ['Separable', 'Low Conditioning', 'High Conditioning', 'Multi-Modal (Global)', 'Multi-Modal (Weak)']
        active_classes = [c for c in standard_classes if c in set(BBOB_CLASSES.get(p, '') for p in PROBLEM_IDS)]
        if not active_classes: active_classes = standard_classes
        
        for s in solvers_list:
            df_s = df_c[df_c['solver_name'] == s] if not df_c.empty else pd.DataFrame()
            if not df_s.empty:
                df_s_map = dict(zip(df_s['hardness_class'], df_s['success_rate']))
                y_vals = [df_s_map.get(c, 0.0) for c in active_classes]
            else:
                y_vals = [0.0 for _ in active_classes]
            
            clean_name = s.split(' / ')[-1].strip().capitalize() if ' / ' in s else s
            fig.add_trace(
                go.Bar(
                    x=active_classes, y=y_vals, name=clean_name,
                    marker_color=SOLVER_COLORS.get(s, '#7f7f7f'),
                    showlegend=(c_idx == 1)
                ),
                row=1, col=c_idx
            )
        
        fig.update_yaxes(title='<b>Success Rate (%)</b>' if c_idx == 1 else None, range=[0, 105], row=1, col=c_idx)
        fig.update_xaxes(title='<b>Landscape Hardness Category</b>', tickangle=-20, row=1, col=c_idx)
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False, row=1, col=c_idx)
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False, row=1, col=c_idx)
    
    fig.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>{model_tag} Mean Success Rate by Landscape Hardness & Noise — {dim}D</b>',
            x=0.02, y=0.98,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#333333'),
        margin=dict(l=65, r=30, t=110, b=75),
        width=1100, height=480,
        legend=dict(
            orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5,
            bgcolor='rgba(255,255,255,0.92)', bordercolor='rgba(0,0,0,0.12)', borderwidth=1,
            font=dict(size=10.5)
        )
    )
    out_p = model_fig_dir(model_tag, dim) / 'figure_success_rate_by_hardness.png'
    fig.write_image(str(out_p), scale=3)

for model_tag, solvers in MODELS_TO_SOLVERS.items():
    for dim in all_dims:
        render_model_success_rate_by_hardness(model_tag, solvers, dim)

print('✅ Model-specific success rate by hardness generated for all discovered models.')


2026-08-24 00:24:08 INFO Chromium init'ed with kwargs {}
2026-08-24 00:24:08 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 00:24:08 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpcuy26okc.
2026-08-24 00:24:08 INFO Opening browser.
2026-08-24 00:24:08 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6e8bvome.
2026-08-24 00:24:08 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6e8bvome
2026-08-24 00:24:09 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpcuy26okc/index.html
2026-08-24 00:24:10 INFO Getting tab from queue (has 1)
2026-08-24 00:24:10 INFO Got B877
2026-08-24 00:24:10 INFO Reloading tab B877 before return.
2026-08-24 00:24:10 INFO Putting tab B877 back (queue size: 0).
2026-08-24 00:24:10 INFO Waiting for all cleanups to finish.
2026-08-24 00:24:10 INFO Exiting Kaleido.
2026-08-24 00:24:10 INFO T

✅ Model-specific success rate by hardness generated for all discovered models.


# 🎓 Part II: Thesis Visual Storyboard (RQ1 → RQ2 → RQ3 → Scaffolding Narrative Chain)

The following four figures form the core visual evidence for the thesis, saved into `results/figures/{dim}D/thesis/`:
- **Figure 1 (RQ1):** Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2):** LLaMEA Synthesis Competency vs. Classical Baselines (Clean Convergence Trajectories & IQR).
- **Figure 3 (RQ3 Hero):** Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation):** Prompt Scaffolding Ablation on LLaMEA-14B (Baseline vs. Guided vs. Thinking vs. Vectorization).


In [24]:
# ── THESIS Figure 1: Benchmark Validation (RQ1: Stochastic Extension) ─────────
for dim in all_dims:
    clean_medians = []
    noisy_medians = []
    problem_labels = []
    
    for p_id in PROBLEM_IDS:
        p_name = BBOB_NAMES.get(p_id, f'f{p_id}')
        p_class = BBOB_CLASSES.get(p_id, '')
        problem_labels.append(f'<b>{p_name}</b><br><sup>{p_class}</sup>')
        
        c_key = (dim, clean_std, p_id)
        c_finals = []
        if c_key in all_benchmark_data:
            for s, runs in all_benchmark_data[c_key].items():
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        c_finals.append(raw_vals[-1])
        clean_medians.append(np.median(c_finals) if c_finals else 1e-16)
        
        n_key = (dim, noisy_std, p_id)
        n_finals = []
        if n_key in all_benchmark_data:
            for s, runs in all_benchmark_data[n_key].items():
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        n_finals.append(raw_vals[-1])
        noisy_medians.append(np.median(n_finals) if n_finals else 1e-16)

    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        name=f'<b>Clean Evaluation (σ={clean_std})</b>',
        x=problem_labels,
        y=np.maximum(clean_medians, 1e-16),
        marker=dict(color='#2B5C8F', line=dict(color='#1B3A5B', width=1.5))
    ))
    fig1.add_trace(go.Bar(
        name=f'<b>Noisy Evaluation (σ={noisy_std})</b>',
        x=problem_labels,
        y=np.maximum(noisy_medians, 1e-16),
        marker=dict(
            color='#D95F02',
            pattern=dict(shape='/', fillmode='replace', fgcolor='#FFFFFF', fgopacity=0.35, size=8),
            line=dict(color='#8C3800', width=1.5)
        )
    ))

    fig1.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Figure 1: Benchmark Problem Difficulty Under Stochastic Noise Extension — {dim}D</b><br><sup>Median Terminal Optimization Precision (Δy) Across Solvers by Landscape Class</sup>',
            x=0.02, y=0.96,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        xaxis=dict(title='<b>BBOB Landscape Class</b>', tickfont=dict(size=11)),
        yaxis=dict(
            type='log', title='<b>Median Final Error log₁₀(Δy)</b>', range=[-16, 4],
            showgrid=True, gridwidth=1, gridcolor='#EAEAEA'
        ),
        barmode='group', bargap=0.25, bargroupgap=0.1,
        width=950, height=540,
        margin=dict(l=65, r=30, t=95, b=65),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=12, color='#333333'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
        )
    )

    out_p = comparative_dir(dim) / 'figure_1_benchmark_validation.png'
    fig1.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 1 generated for all dimensions in results/publication/{dim}D/.')


2026-08-24 00:24:21 INFO Chromium init'ed with kwargs {}
2026-08-24 00:24:21 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 00:24:21 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppikeljqp.
2026-08-24 00:24:21 INFO Opening browser.
2026-08-24 00:24:21 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpnoydgj8y.
2026-08-24 00:24:21 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpnoydgj8y
2026-08-24 00:24:23 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppikeljqp/index.html
2026-08-24 00:24:23 INFO Getting tab from queue (has 1)
2026-08-24 00:24:23 INFO Got F66C
2026-08-24 00:24:23 INFO Reloading tab F66C before return.
2026-08-24 00:24:24 INFO Putting tab F66C back (queue size: 0).
2026-08-24 00:24:24 INFO Waiting for all cleanups to finish.
2026-08-24 00:24:24 INFO Exiting Kaleido.
2026-08-24 00:24:24 INFO T

✅ Thesis Figure 1 generated for all dimensions in results/figures/comparative/{dim}D/.


In [25]:
# ── THESIS Figure 2: Empirical Convergence Trajectories & ECDFs (RQ2 & RQ3) ──
eval_grid = np.logspace(0, 5, 200)
targets = np.logspace(-8, 2, 100)

def get_dynamic_grid(n_problems: int, max_cols: int = 3):
    n_total = n_problems + 1  # problems + 1 overall ECDF
    n_cols = min(max_cols, n_total)
    n_rows = int(np.ceil(n_total / n_cols))
    coords = [((i // n_cols) + 1, (i % n_cols) + 1) for i in range(n_total)]
    return coords, n_rows, n_cols

def render_figure_2_trajectories(model_slug: str, solvers_to_plot: list, dim: int, noise_std: float, label_env: str):
    coords, n_rows, n_cols = get_dynamic_grid(len(PROBLEM_IDS), max_cols=3)
    subplot_titles = [
        f"<b>{BBOB_NAMES.get(p, f'f{p}')}</b><br><sup>{BBOB_CLASSES.get(p, '')}</sup>"
        for p in PROBLEM_IDS
    ] + ['<b>Overall Benchmark Portfolio</b><br><sup>Empirical Target Hit Rate (ECDF)</sup>']

    fig_comp = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.22 if n_rows <= 2 else 0.15
    )

    for idx, p_id in enumerate(PROBLEM_IDS):
        r_idx, c_idx = coords[idx]
        key = (dim, noise_std, p_id)
        if key not in all_benchmark_data: continue
        solvers_data = all_benchmark_data[key]
        
        for s in solvers_to_plot:
            if s not in solvers_data or not solvers_data[s]: continue
            runs = solvers_data[s]
            interpolated = [np.interp(eval_grid, evals, raw_vals, left=raw_vals[0], right=raw_vals[-1]) for evals, raw_vals in runs if len(evals) > 0]
            if not interpolated: continue
            
            arr = np.array(interpolated)
            med = np.median(arr, axis=0)
            q25 = np.percentile(arr, 25, axis=0)
            q75 = np.percentile(arr, 75, axis=0)
            
            col = SOLVER_COLORS.get(s, '#7f7f7f')
            hex_c = col.lstrip('#')
            rgb = tuple(int(hex_c[i:i+2], 16) for i in (0, 2, 4))
            rgba_fill = f'rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.15)'
            is_llm = ' / ' in s
            grp_id = s.split(' / ')[0] if is_llm else 'Classical'
            show_leg = (idx == 0)
            
            # IQR shaded band
            fig_comp.add_trace(
                go.Scatter(
                    x=np.concatenate([eval_grid, eval_grid[::-1]]),
                    y=np.concatenate([np.maximum(q75, 1e-16), np.maximum(q25[::-1], 1e-16)]),
                    fill='toself', fillcolor=rgba_fill,
                    line=dict(color='rgba(255,255,255,0)'),
                    hoverinfo='skip', showlegend=False
                ),
                row=r_idx, col=c_idx
            )
            
            # Median line
            fig_comp.add_trace(
                go.Scatter(
                    x=eval_grid, y=np.maximum(med, 1e-16), mode='lines', name=s,
                    legendgroup=grp_id,
                    line=dict(color=col, width=2.6 if is_llm else 1.6, dash='solid' if is_llm else 'dash'),
                    showlegend=show_leg
                ),
                row=r_idx, col=c_idx
            )
            
        fig_comp.add_hline(y=1e-8, line_dash='dot', line_color='rgba(0,0,0,0.3)', row=r_idx, col=c_idx)
        fig_comp.update_xaxes(type='log', title='<b>Evaluations</b>', row=r_idx, col=c_idx)
        fig_comp.update_yaxes(type='log', range=[-16, 5], title='<b>Precision log₁₀(Δy)</b>' if c_idx == 1 else None, row=r_idx, col=c_idx)

    # Subplot for Overall Portfolio ECDF
    ecdf_r, ecdf_c = coords[-1]
    for s in solvers_to_plot:
        all_s_terminals = []
        for p_id in PROBLEM_IDS:
            key = (dim, noise_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                all_s_terminals.extend(finals)
        if not all_s_terminals: continue
        
        hit_rates = [np.mean(np.array(all_s_terminals) <= t) for t in targets]
        is_llm = ' / ' in s
        col = SOLVER_COLORS.get(s, '#7f7f7f')
        fig_comp.add_trace(
            go.Scatter(
                x=targets, y=hit_rates, mode='lines', name=s,
                line=dict(color=col, width=2.6 if is_llm else 1.6, dash='solid' if is_llm else 'dash'),
                showlegend=False
            ),
            row=ecdf_r, col=ecdf_c
        )
    fig_comp.update_xaxes(type='log', title='<b>Target Precision (Δy)</b>', row=ecdf_r, col=ecdf_c)
    fig_comp.update_yaxes(title='<b>Portfolio Hit Rate</b>', range=[0, 1.05], row=ecdf_r, col=ecdf_c)

    fig_comp.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Empirical Convergence Trajectories & Portfolio Competency ({label_env}) — {dim}D</b>',
            x=0.02, y=0.985,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#333333'),
        margin=dict(l=65, r=30, t=140 if n_rows <= 2 else 160, b=55),
        width=1220, height=430 * n_rows,
        legend=dict(
            orientation='h', yanchor='bottom', y=1.03, xanchor='center', x=0.5,
            bgcolor='rgba(255,255,255,0.92)', bordercolor='rgba(0,0,0,0.12)', borderwidth=1,
            font=dict(size=10.5), groupclick='toggleitem'
        )
    )
    fig_comp.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False)
    fig_comp.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False)

    out_dir = model_std_dir(model_slug, dim, noise_std)
    out_p = out_dir / 'convergence_trajectories.png'
    fig_comp.write_image(str(out_p), scale=3)

def render_figure_2_ecdf(model_slug: str, solvers_to_plot: list, dim: int, noise_std: float, label_env: str):
    coords, n_rows, n_cols = get_dynamic_grid(len(PROBLEM_IDS), max_cols=3)
    subplot_titles = [
        f"<b>{BBOB_NAMES.get(p, f'f{p}')}</b><br><sup>{BBOB_CLASSES.get(p, '')}</sup>"
        for p in PROBLEM_IDS
    ] + ['<b>Overall Benchmark Portfolio</b><br><sup>Empirical Target Hit Rate (All Problems)</sup>']

    fig_ecdf = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.22 if n_rows <= 2 else 0.15
    )

    for idx, p_id in enumerate(PROBLEM_IDS):
        r_idx, c_idx = coords[idx]
        key = (dim, noise_std, p_id)
        if key not in all_benchmark_data: continue
        solvers_data = all_benchmark_data[key]
        
        for s in solvers_to_plot:
            if s not in solvers_data or not solvers_data[s]: continue
            runs = solvers_data[s]
            finals = [r[1][-1] for r in runs if len(r[1]) > 0]
            if not finals: continue
            
            hit_rates = [np.mean(np.array(finals) <= t) for t in targets]
            col = SOLVER_COLORS.get(s, '#7f7f7f')
            is_llm = ' / ' in s
            grp_id = s.split(' / ')[0] if is_llm else 'Classical Baselines'
            show_leg = (idx == 0)
            first_in_grp = (s == [x for x in solvers_to_plot if (x.split(' / ')[0] if ' / ' in x else 'Classical Baselines') == grp_id][0])
            
            fig_ecdf.add_trace(
                go.Scatter(
                    x=targets, y=hit_rates, mode='lines', name=s,
                    legendgroup=grp_id,
                    legendgrouptitle_text=f'<b>{grp_id}</b>' if first_in_grp else None,
                    line=dict(color=col, width=2.6 if is_llm else 1.6, dash='solid' if is_llm else 'dash'),
                    showlegend=show_leg
                ),
                row=r_idx, col=c_idx
            )
        fig_ecdf.update_xaxes(type='log', title='<b>Target Precision (Δy)</b>', row=r_idx, col=c_idx)
        fig_ecdf.update_yaxes(title='<b>Hit Rate (ECDF)</b>' if c_idx == 1 else None, range=[0, 1.05], row=r_idx, col=c_idx)

    # Subplot for Overall Portfolio ECDF
    ecdf_r, ecdf_c = coords[-1]
    for s in solvers_to_plot:
        all_s_terminals = []
        for p_id in PROBLEM_IDS:
            key = (dim, noise_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                finals = [r[1][-1] for r in runs if len(r[1]) > 0]
                all_s_terminals.extend(finals)
        if not all_s_terminals: continue
        
        hit_rates = [np.mean(np.array(all_s_terminals) <= t) for t in targets]
        is_llm = ' / ' in s
        col = SOLVER_COLORS.get(s, '#7f7f7f')
        fig_ecdf.add_trace(
            go.Scatter(
                x=targets, y=hit_rates, mode='lines', name=s,
                line=dict(color=col, width=2.6 if is_llm else 1.6, dash='solid' if is_llm else 'dash'),
                showlegend=False
            ),
            row=ecdf_r, col=ecdf_c
        )
    fig_ecdf.update_xaxes(type='log', title='<b>Target Precision (Δy)</b>', row=ecdf_r, col=ecdf_c)
    fig_ecdf.update_yaxes(title='<b>Portfolio Hit Rate</b>', range=[0, 1.05], row=ecdf_r, col=ecdf_c)

    fig_ecdf.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Empirical Target Precision Hit Rates (ECDF) ({label_env}) — {dim}D</b>',
            x=0.02, y=0.985,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=11, color='#333333'),
        margin=dict(l=65, r=30, t=140 if n_rows <= 2 else 160, b=55),
        width=1220, height=430 * n_rows,
        legend=dict(
            orientation='h', yanchor='bottom', y=1.03, xanchor='center', x=0.5,
            bgcolor='rgba(255,255,255,0.92)', bordercolor='rgba(0,0,0,0.12)', borderwidth=1,
            font=dict(size=10.5), groupclick='toggleitem'
        )
    )
    fig_ecdf.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False)
    fig_ecdf.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EAEAEA', zeroline=False)

    out_dir = model_std_dir(model_slug, dim, noise_std)
    out_p = out_dir / 'target_precision_ecdf.png'
    fig_ecdf.write_image(str(out_p), scale=3)

# Render trajectory and ECDF figures for each model family + classical baselines
for model_tag, m_solvers in MODELS_TO_SOLVERS.items():
    solvers_to_plot = m_solvers + SOLVERS_CLASSICAL
    for dim in all_dims:
        for n_std in all_noise_stds:
            env_name = 'Clean' if np.isclose(n_std, 0.0) else 'Noisy'
            render_figure_2_trajectories(model_tag, solvers_to_plot, dim, n_std, f'{env_name} σ={n_std}')
            render_figure_2_ecdf(model_tag, solvers_to_plot, dim, n_std, f'{env_name} σ={n_std}')

print('✅ Empirical Trajectories and ECDFs generated dynamically for all models and dimensions.')


2026-08-24 00:24:28 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 00:24:28 INFO shutil.rmtree worked.
2026-08-24 00:24:28 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 00:24:28 INFO shutil.rmtree worked.
2026-08-24 00:24:28 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 00:24:28 INFO shutil.rmtree worked.
2026-08-24 00:24:28 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 00:24:28 INFO shutil.rmtree worked.
2026-08-24 00:24:28 INFO Chromium init'ed with kwargs {}
2026-08-24 00:24:28 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 00:24:28 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpps8qzm12.
2026-08-24 00:24:28 INFO Opening browser.
2026-08-24 00:24:28 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpv8fesug6.
2026-08-24 00:24:28 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpv8fesug6
2026-08-24 00:24:29 INFO C

✅ Empirical Trajectories and ECDFs generated dynamically for all models and dimensions.


In [26]:
# ── THESIS Figure 3: Cross-Environment Noise Robustness Profile (RQ3 Hero) ────
for dim in all_dims:
    fig_rob = go.Figure()
    solvers_in_plot = [s for s in ALL_SOLVERS_ORDER]
    clean_rates = []
    noisy_rates = []
    valid_solvers = []
    
    for s in solvers_in_plot:
        c_succ, c_tot = 0, 0
        for p_id in PROBLEM_IDS:
            key = (dim, clean_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        c_tot += 1
                        if raw_vals[-1] <= 1e-8:
                            c_succ += 1
        
        n_succ, n_tot = 0, 0
        for p_id in PROBLEM_IDS:
            key = (dim, noisy_std, p_id)
            if key in all_benchmark_data and s in all_benchmark_data[key]:
                runs = all_benchmark_data[key][s]
                for evals, raw_vals in runs:
                    if len(raw_vals) > 0:
                        n_tot += 1
                        if raw_vals[-1] <= 1e-8:
                            n_succ += 1
                            
        if c_tot > 0 and n_tot > 0:
            clean_rates.append(c_succ / c_tot * 100.0)
            noisy_rates.append(n_succ / n_tot * 100.0)
            valid_solvers.append(s)

    if not valid_solvers: continue

    # Add vertical reference drops (connectors)
    for idx, s in enumerate(valid_solvers):
        c_val = clean_rates[idx]
        n_val = noisy_rates[idx]
        drop = c_val - n_val
        line_color = '#2CA02C' if drop <= 10 else ('#D62728' if drop >= 25 else '#7F7F7F')
        
        fig_rob.add_trace(go.Scatter(
            x=[n_val, c_val], y=[s, s], mode='lines',
            line=dict(color=line_color, width=3.5),
            hoverinfo='skip', showlegend=False
        ))

    # Add Clean markers (Open Circle)
    fig_rob.add_trace(go.Scatter(
        x=clean_rates, y=valid_solvers, mode='markers',
        name=f'<b>Clean Success Rate (σ={clean_std})</b>',
        marker=dict(symbol='circle-open', size=14, color='#2B5C8F', line=dict(width=2.5, color='#2B5C8F'))
    ))

    # Add Noisy markers (Filled Circle)
    fig_rob.add_trace(go.Scatter(
        x=noisy_rates, y=valid_solvers, mode='markers',
        name=f'<b>Noisy Success Rate (σ={noisy_std})</b>',
        marker=dict(symbol='circle', size=14, color='#D95F02', line=dict(width=1.5, color='#8C3800'))
    ))

    fig_rob.update_layout(
        template='plotly_white',
        title=dict(
            text=f'<b>Figure 3: Cross-Environment Algorithm Robustness & Degradation Profile — {dim}D</b><br><sup>Paired Comparison of Precision Success Rate (Δy ≤ 10⁻⁸) on Clean (σ={clean_std}) vs. Noisy (σ={noisy_std}) Landscapes</sup>',
            x=0.02, y=0.96,
            font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
        ),
        xaxis=dict(title='<b>Success Rate (%)</b>', range=[-2, 105], showgrid=True, gridwidth=1, gridcolor='#EAEAEA'),
        yaxis=dict(title='<b>Optimization Solver</b>', autorange='reversed', tickfont=dict(size=11)),
        width=980, height=520,
        margin=dict(l=190, r=40, t=95, b=65),
        font=dict(family='Inter, Helvetica, Arial, sans-serif', size=12, color='#333333'),
        legend=dict(
            orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
        )
    )

    out_p = comparative_dir(dim) / 'figure_3_robustness.png'
    fig_rob.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 3 generated for all dimensions in results/publication/{dim}D/.')


2026-08-24 00:25:22 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 00:25:22 INFO shutil.rmtree worked.
2026-08-24 00:25:22 INFO Chromium init'ed with kwargs {}
2026-08-24 00:25:22 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 00:25:22 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6qifb3sm.
2026-08-24 00:25:22 INFO Opening browser.
2026-08-24 00:25:22 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpirqppk8j.
2026-08-24 00:25:22 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpirqppk8j
2026-08-24 00:25:23 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp6qifb3sm/index.html
2026-08-24 00:25:24 INFO Getting tab from queue (has 1)
2026-08-24 00:25:24 INFO Got FC1E
2026-08-24 00:25:24 INFO Reloading tab FC1E before return.
2026-08-24 00:25:24 INFO Putting tab FC1E back (queue size: 0).
2026-08-24 00:25:24 

✅ Thesis Figure 3 generated for all dimensions in results/figures/comparative/{dim}D/.


In [27]:
# ── THESIS Figure 4: Prompt Scaffolding Ablation Across Discovered Models (RQ2/3) ──
for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        if len(solvers_list) <= 1:
            continue  # Skip single-strategy models

        strategies = [s.split(' / ')[1] for s in solvers_list]
        strat_labels = [f'<b>{s.title()}</b>' for s in strategies]
        
        clean_succ_rates = []
        noisy_succ_rates = []
        
        for s_name in solvers_list:
            c_succ, c_tot = 0, 0
            for p_id in PROBLEM_IDS:
                key = (dim, clean_std, p_id)
                if key in all_benchmark_data and s_name in all_benchmark_data[key]:
                    runs = all_benchmark_data[key][s_name]
                    for evals, raw_vals in runs:
                        if len(raw_vals) > 0:
                            c_tot += 1
                            if raw_vals[-1] <= 1e-8:
                                c_succ += 1
            clean_succ_rates.append(c_succ / c_tot * 100.0 if c_tot > 0 else 0.0)
            
            n_succ, n_tot = 0, 0
            for p_id in PROBLEM_IDS:
                key = (dim, noisy_std, p_id)
                if key in all_benchmark_data and s_name in all_benchmark_data[key]:
                    runs = all_benchmark_data[key][s_name]
                    for evals, raw_vals in runs:
                        if len(raw_vals) > 0:
                            n_tot += 1
                            if raw_vals[-1] <= 1e-8:
                                n_succ += 1
            noisy_succ_rates.append(n_succ / n_tot * 100.0 if n_tot > 0 else 0.0)

        fig4 = go.Figure()
        fig4.add_trace(go.Bar(
            name=f'<b>Clean (σ={clean_std})</b>',
            x=strat_labels, y=clean_succ_rates,
            marker=dict(color='#2B5C8F', line=dict(color='#1B3A5B', width=1.5))
        ))
        fig4.add_trace(go.Bar(
            name=f'<b>Noisy (σ={noisy_std})</b>',
            x=strat_labels, y=noisy_succ_rates,
            marker=dict(
                color='#D95F02',
                pattern=dict(shape='/', fillmode='replace', fgcolor='#FFFFFF', fgopacity=0.35, size=8),
                line=dict(color='#8C3800', width=1.5)
            )
        ))

        fig4.update_layout(
            template='plotly_white',
            title=dict(
                text=f'<b>Figure 4: Impact of Prompt Scaffolding on {model_name} Synthesis Quality — {dim}D</b><br><sup>Precision Success Rate (Δy ≤ 10⁻⁸) Across Clean vs. Stochastic Benchmark Regimes</sup>',
                x=0.02, y=0.96,
                font=dict(size=14, color='#2c3e50', family='Inter, Helvetica, Arial, sans-serif')
            ),
            xaxis=dict(title='<b>Prompt Scaffolding Strategy</b>', tickfont=dict(size=11)),
            yaxis=dict(title='<b>Benchmark Success Rate (%)</b>', range=[0, 105], showgrid=True, gridwidth=1, gridcolor='#EAEAEA'),
            barmode='group', bargap=0.28, bargroupgap=0.1,
            width=880, height=500,
            margin=dict(l=65, r=30, t=95, b=65),
            font=dict(family='Inter, Helvetica, Arial, sans-serif', size=12, color='#333333'),
            legend=dict(
                orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1.0,
                bgcolor='rgba(255,255,255,0.9)', bordercolor='rgba(0,0,0,0.15)', borderwidth=1
            )
        )

        out_p = model_fig_dir(model_name, dim) / 'figure_4_scaffolding.png'
        fig4.write_image(str(out_p), scale=3)

print('✅ Thesis Figure 4 generated dynamically for all multi-strategy models.')


2026-08-24 00:25:28 INFO TemporaryDirectory.cleanup() worked.
2026-08-24 00:25:28 INFO shutil.rmtree worked.
2026-08-24 00:25:28 INFO Chromium init'ed with kwargs {}
2026-08-24 00:25:28 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-24 00:25:28 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmps65g4dee.
2026-08-24 00:25:28 INFO Opening browser.
2026-08-24 00:25:28 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpsw1jqxdk.
2026-08-24 00:25:28 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpsw1jqxdk
2026-08-24 00:25:29 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmps65g4dee/index.html
2026-08-24 00:25:30 INFO Getting tab from queue (has 1)
2026-08-24 00:25:30 INFO Got 90F8
2026-08-24 00:25:30 INFO Reloading tab 90F8 before return.
2026-08-24 00:25:30 INFO Putting tab 90F8 back (queue size: 0).
2026-08-24 00:25:30 

✅ Thesis Figure 4 generated dynamically for all multi-strategy models.
